# Errors, Diagnostics & Troubleshooting — dask_setup Recipe

This notebook demonstrates the most common failure modes when using `dask_setup` and
shows concrete solutions for each:

1. Memory limit exceeded (`Task needs > memory_limit`)
2. OOM crashes and memory spilling tuning
3. Workers failing to start
4. Bad chunk sizes (too small → overhead, too large → OOM)
5. Dashboard access on HPC
6. Resource detection failures
7. Diagnosing slowdowns with the Dask dashboard and `cluster_report()`

## Setup

In [ ]:
import os
import time
from contextlib import contextmanager

import dask.array as da
import numpy as np
import xarray as xr

from dask_setup import setup_dask_client
from dask_setup.config import DaskSetupConfig


@contextmanager
def demo_cluster(label="", **kwargs):
    """Context manager that creates and cleans up a cluster."""
    defaults = dict(fallback_on_detection_failure=True, dashboard=False)
    defaults.update(kwargs)
    client, cluster, tmp = setup_dask_client(**defaults)
    print(f"[{label}] {len(client.scheduler_info()['workers'])} workers, tmp={tmp}")
    try:
        yield client, cluster, tmp
    finally:
        client.close()
        cluster.close()

print("Imports OK")

## 1. Memory Limit Exceeded

**Symptom:** `distributed.worker - WARNING - Memory use is high ... Task needs > memory_limit`

This happens when a single task is larger than the per-worker memory allocation.

**Solutions:**  
- Reduce chunk sizes  
- Reduce the number of workers (more memory per worker)  
- Lower `reserve_mem_gb`

In [ ]:
# PROBLEM: many small workers → each has very little memory
# Fix: fewer workers so each gets more memory budget

with demo_cluster("memory-fix", workload_type="cpu", max_workers=2, reserve_mem_gb=2.0) as (client, _, _):
    # Moderate-size array — will fit comfortably with 2 workers
    x = da.random.random((3000, 3000), chunks=(500, 500))
    result = x.mean().compute()
    print(f"Result: {result:.6f}")
    print("Solution: use fewer workers and/or smaller chunks")

## 2. Tuning Memory Spill Thresholds

Dask spills to disk when memory pressure reaches configurable thresholds.
Lower `memory_target` to start spilling earlier and avoid OOM crashes.

In [ ]:
# Conservative configuration for memory-constrained systems
conservative_config = DaskSetupConfig(
    workload_type="cpu",
    max_workers=2,
    reserve_mem_gb=2.0,
    memory_target=0.60,   # start spilling at 60% usage
    memory_spill=0.70,    # aggressive spill at 70%
    memory_pause=0.85,    # pause new tasks at 85%
    memory_terminate=0.95,
    spill_compression="lz4",  # fast spill compression
)

print("Conservative memory config:")
print(f"  memory_target   : {conservative_config.memory_target}")
print(f"  memory_spill    : {conservative_config.memory_spill}")
print(f"  spill_compression: {conservative_config.spill_compression}")

with demo_cluster("conservative", config=conservative_config) as (client, _, tmp):
    x = da.random.random((2000, 2000), chunks=(400, 400))
    result = x.mean().compute()
    print(f"  Completed with conservative thresholds — result={result:.4f}")

## 3. Bad Chunk Sizes

Chunks that are too small cause massive scheduler overhead (millions of tiny tasks).
Chunks that are too large cause OOM. The target is **256–512 MiB per chunk**.

In [ ]:
# Create a sample dataset
ds = xr.Dataset(
    {"temperature": (["time", "lat", "lon"],
                     np.random.random((200, 90, 180)).astype("float32"))},
    coords={
        "time": np.arange(200),
        "lat": np.linspace(-90, 90, 90),
        "lon": np.linspace(-180, 180, 180),
    },
)

dtype_bytes = ds.temperature.values.itemsize

def chunk_mb(chunks):
    import math
    elements = math.prod(chunks.values())
    return elements * dtype_bytes / 1e6

chunk_options = {
    "too_small":  {"time": 5,   "lat": 10, "lon": 10},
    "optimal":    {"time": 50,  "lat": 90, "lon": 180},
    "too_large":  {"time": 200, "lat": 90, "lon": 180},
}

print(f"{'Strategy':<12}  {'Chunk MB':>10}  {'n_chunks':>8}  {'Note'}")
print("-" * 55)
for label, chunks in chunk_options.items():
    mb = chunk_mb(chunks)
    ds_c = ds.chunk(chunks)
    n = ds_c.temperature.data.npartitions
    ok = "✅ optimal" if 128 <= mb <= 512 else ("⚠️ too small" if mb < 128 else "⚠️ too large")
    print(f"{label:<12}  {mb:>10.1f}  {n:>8}  {ok}")

In [ ]:
# Let dask_setup choose chunk sizes for you
from dask_setup import recommend_chunks

with demo_cluster("chunk-advisor", workload_type="cpu", max_workers=2,
                  reserve_mem_gb=2.0) as (client, _, _):
    suggested = recommend_chunks(ds, client, verbose=True)
    print(f"Recommended chunks: {suggested}")

## 4. Workers Fail to Start

On some HPC systems, worker processes are killed at startup if they exceed scheduler resource
limits. Fix by reducing `max_workers` or using `workload_type="io"` (single process).

In [ ]:
# io topology = single process, many threads → no per-process startup cost
with demo_cluster("io-fallback", workload_type="io", reserve_mem_gb=2.0) as (client, _, _):
    workers = client.scheduler_info()["workers"]
    first_w = next(iter(workers.values()))
    print(f"Workers: {len(workers)}")
    print(f"Threads per worker: {first_w.get('nthreads', '?')}")
    print("Single-process / multi-thread topology avoids startup issues.")

## 5. Resource Detection Failures

`dask_setup` reads `$PBS_JOBFS`, `$NCPUS`, `$PBS_MEM` and `psutil`.
If all fail, set `fallback_on_detection_failure=True` to get a safe 2-core, 8 GiB default.

In [ ]:
print("HPC environment variables:")
for var in ["PBS_JOBID", "PBS_JOBFS", "NCPUS", "PBS_MEM",
            "SLURM_JOB_ID", "SLURM_CPUS_ON_NODE", "SLURM_MEM_PER_NODE"]:
    val = os.environ.get(var)
    if val:
        print(f"  {var}={val}")

# Safe fallback for unknown environments
with demo_cluster("safe-fallback", workload_type="cpu",
                  fallback_on_detection_failure=True) as (client, _, tmp):
    print(f"Workers: {len(client.scheduler_info()['workers'])}  tmp={tmp}")
    print("fallback_on_detection_failure=True prevents errors on unknown systems.")

## 6. Dashboard on HPC — SSH Tunnel

The Dask dashboard is not directly accessible from a login node. Use the SSH tunnel
command printed by `setup_dask_client()` to forward the port:

In [ ]:
import socket

client, cluster, tmp = setup_dask_client(
    workload_type="cpu",
    max_workers=2,
    reserve_mem_gb=2.0,
    dashboard=True,
    fallback_on_detection_failure=True,
)

if hasattr(cluster, "dashboard_link") and cluster.dashboard_link:
    port = cluster.dashboard_link.split(":")[-1].split("/")[0]
    hostname = socket.gethostname()
    print(f"Dashboard: {cluster.dashboard_link}")
    print(f"SSH tunnel: ssh -N -L 8787:{hostname}:{port} <login_node>")
    print("Then open http://localhost:8787 in your browser.")

client.close()
cluster.close()

## 7. Post-Run Diagnostics with `cluster_report()`

`cluster_report()` collects peak memory, spill totals, and task throughput — useful for
identifying bottlenecks without leaving the notebook.

In [ ]:
from dask_setup.reporting import cluster_report

with demo_cluster("diagnostics", workload_type="cpu", max_workers=2,
                  reserve_mem_gb=2.0) as (client, _, _):
    # Run some work
    x = da.random.random((2000, 2000), chunks=(500, 500))
    _ = (x + x.T).mean().compute()

    # Collect diagnostics
    report = cluster_report(client)
    print(f"Peak memory  : {report.peak_memory_gib:.3f} GiB")
    print(f"Total spill  : {report.total_spill_gib:.3f} GiB")

## Quick Reference — Common Fixes

| Symptom | Most likely cause | Quick fix |
|---------|------------------|-----------|
| `Task needs > memory_limit` | Chunk too large for worker | Reduce chunks or use `max_workers=1` |
| Worker killed / KilledWorker | OOM | Lower `memory_target`, increase job RAM |
| Very slow performance | Too many tiny tasks | Increase chunk size to 256–512 MiB |
| Workers fail to start | HPC process limits | Use `workload_type="io"` or reduce `max_workers` |
| Spill goes to `/scratch` | `$PBS_JOBFS` not set | Add `#PBS -l jobfs=200gb` to job script |
| Dashboard unreachable | Behind compute node firewall | Use SSH tunnel from the startup output |
| `reserve_mem_gb` too large | Small machine | Use `reserve_mem_gb=4.0` or `profile="development"` |

See the [Troubleshooting wiki page](https://github.com/21centuryweather/dask_setup/wiki/Troubleshooting) for full details.